##### make_interval()

- is used to create an **INTERVAL type** from individual components like **years, months, weeks, days, hours, minutes, and seconds**.
- It is available starting from **Spark version 3.5.0.**

##### Syntax

     make_interval(years=<years>, months=<months>, weeks=<weeks>, days=<days>, hours=<hours>, mins=<mins>, secs=<secs>)

     # Correct order
     make_interval(years, months, weeks, days, hours, mins, secs)

##### Parameters

| parameter |          Description                              |
|-----------|---------------------------------------------------|
|   years   | The number of years, positive or negative.        |
|   months  | The number of months, positive or negative.       |
|   weeks   | The number of weeks, positive or negative.        |
|   days    | The number of days, positive or negative.         |
|   hours   | The number of hours, positive or negative.        |
|   mins    | The number of minutes, positive or negative.      |
|   secs    | The number of seconds, with fractional parts in microsecond precision. | 

**Return**
- A **new column** that contains an **interval**.

In [0]:
from pyspark.sql.functions import lit, col, expr, make_interval, to_date, add_months

##### 1) Create an Interval from `Years, Months, Week, Days, Hour, Minute, Second`

In [0]:
df1 = spark.range(1)

df1.select(
    make_interval(
        lit(1),      # years
        lit(2),      # months
        lit(3),      # weeks (7x3=21 days)
        lit(4),      # days  21 + 4 = 25 days
        lit(5),      # hours
        lit(6),      # minutes
        lit(7.123)   # seconds (can be decimal)
    ).alias("interval_value"),
    make_interval(
        lit(1),      # years
        lit(2),      # months
        lit(3),      # weeks (7x3=21 days)
        lit(4),      # days  21 + 4 = 25 days
        lit(5),      # hours
        lit(6),      # minutes
        lit(None)    # seconds (can be decimal)
    ).alias("interval_value_null"),
    make_interval().alias("interval_null"),
).display()

interval_value,interval_value_null,interval_null
1 years 2 months 25 days 5 hours 6 minutes 7.123 seconds,null,0 seconds


##### 2) Create an interval from a column in a DataFrame

In [0]:
df2 = spark.createDataFrame([[1, 15]], ["days", "minutes"])

df2.select("days", "minutes", make_interval(days=df2["days"], mins=df2["minutes"]).alias("days_minutes_interval")).display()

days,minutes,days_minutes_interval
1,15,1 days 15 minutes


In [0]:
# Example with an overflow that might cause an error
df3 = spark.createDataFrame([[5, 2, 30, 10]], ["years", "months", "weeks", "days"])

df3.select("years", "months", "weeks", "days", try_make_interval(df3.years, df3.months, df3.weeks, df3.days).alias("interval_years_months")).display()

years,months,weeks,days,interval_years_months
5,2,30,10,5 years 2 months 220 days


In [0]:
df4 = spark.createDataFrame([[10, 11, 1, 1, 12, 30, 1.001001]], ['year', 'month', 'week', 'day', 'hour', 'min', 'sec'])
display(df4)

df4_final = df4.select(
    make_interval(df4.year, df4.month, df4.week, df4.day, df4.hour, df4.min, df4.sec).alias("interval_full"),
    make_interval(df4.year, df4.month, df4.week, df4.day, df4.hour, df4.min).alias("interval_sec"),
    make_interval(df4.year, df4.month, df4.week, df4.day, df4.hour).alias("interval_min_sec"),
    make_interval(df4.year, df4.month, df4.week, df4.day).alias("interval_time"),
    make_interval(df4.year, df4.month, df4.week).alias("interval_time_day"),
    make_interval(df4.year, df4.month).alias("year_month"),
    make_interval(df4.year).alias("year"),
)

df4_final.display()

year,month,week,day,hour,min,sec
10,11,1,1,12,30,1.001001


interval_full,interval_sec,interval_min_sec,interval_time,interval_time_day,year_month,year
10 years 11 months 8 days 12 hours 30 minutes 1.001001 seconds,10 years 11 months 8 days 12 hours 30 minutes,10 years 11 months 8 days 12 hours,10 years 11 months 8 days,10 years 11 months 7 days,10 years 11 months,10 years


##### 3) Handling potential errors with try_make_interval
- If `invalid inputs might occur`, use **try_make_interval** which returns `NULL` instead of `raising an error`.

In [0]:
from pyspark.sql.functions import try_make_interval

# Example with an overflow that might cause an error
df4 = spark.createDataFrame([[100, 100]], ["years", "months"])

df4.select("years", "months", make_interval(df4.years, df4.months).alias("interval_years_months")).display()

years,months,interval_years_months
100,100,108 years 4 months


In [0]:
from pyspark.sql.functions import try_make_interval

# Example with an overflow that might cause an error
df4 = spark.createDataFrame([[100, 100]], ["years", "months"])

df4.select("years", "months", try_make_interval(df4.years, df4.months).alias("interval_years_months")).display()

years,months,interval_years_months
100,100,108 years 4 months


##### 4) Basic Example

In [0]:
# Create DataFrame
data = [
    (1, 0, 0, 0, 0, 0, 0),
    (0, 1, 0, 0, 0, 0, 0),
    (0, 0, 1, 0, 0, 0, 0),
    (1, 8, 0, 0, 0, 0, 0),
    (1, 5, 25, 0, 0, 0, 0),
    (0, 0, 0, 23, 0, 0, 0),
    (0, 0, 0, 0, 45, 0, 0),
    (0, 0, 0, 0, 0, 58, 0),
    (0, 0, 0, 0, 0, 0, 59),
    (1, 10, 25, 22, 45, 55, 47)
]
df_intr = spark.createDataFrame(data, ["years", "months", "weeks", "days", "hours", "minutes", "seconds"])
display(df_intr)

years,months,weeks,days,hours,minutes,seconds
1,0,0,0,0,0,0
0,1,0,0,0,0,0
0,0,1,0,0,0,0
1,8,0,0,0,0,0
1,5,25,0,0,0,0
0,0,0,23,0,0,0
0,0,0,0,45,0,0
0,0,0,0,0,58,0
0,0,0,0,0,0,59
1,10,25,22,45,55,47


In [0]:
# Use make_interval() to create interval
df_intr_final = df_intr.withColumn("interval", make_interval("years", "months", "weeks", "days", "hours", "minutes", "seconds"))
display(df_intr_final)

years,months,weeks,days,hours,minutes,seconds,interval
1,0,0,0,0,0,0,1 years
0,1,0,0,0,0,0,1 months
0,0,1,0,0,0,0,7 days
1,8,0,0,0,0,0,1 years 8 months
1,5,25,0,0,0,0,1 years 5 months 175 days
0,0,0,23,0,0,0,23 days
0,0,0,0,45,0,0,45 hours
0,0,0,0,0,58,0,58 minutes
0,0,0,0,0,0,59,59 seconds
1,10,25,22,45,55,47,1 years 10 months 197 days 45 hours 55 minutes 47 seconds


##### 4) Add Interval to Timestamp
- `Adds 1 month to current timestamp`.

In [0]:
df_ts_intr = spark.createDataFrame([(1,)], ["id"])

df_ts_intr.select(
    make_interval(
        lit(1),   # years
        lit(2),   # months
        lit(5),   # weeks
        lit(10),  # days
        lit(12),  # hours
        lit(30),  # minutes
        lit(20)   # seconds
    ).alias("interval_value")
).display()

interval_value
1 years 2 months 45 days 12 hours 30 minutes 20 seconds


In [0]:
from pyspark.sql.functions import current_timestamp

spark.range(1).select(
    current_timestamp().alias("current_time"),
    (current_timestamp() + make_interval(lit(0), lit(1), lit(0), lit(0), lit(0), lit(0))).alias("after_1_month"),
    (current_timestamp() + make_interval(lit(-1), lit(-6), lit(0), lit(0), lit(0), lit(0))).alias("before_1_year_6_months")
).display()

current_time,after_1_month,before_1_year_6_months
2026-03-17T19:27:40.275Z,2026-04-17T19:27:40.275Z,2024-09-17T19:27:40.275Z


##### 5) Subtract Interval from Date
- `Subtracts 7 days.`

In [0]:
from pyspark.sql.functions import current_date

spark.range(1).select(
    current_date().alias("today"),
    (current_date() - make_interval(lit(0), lit(0), lit(0), lit(7), lit(0), lit(0))).alias("before_7_days")
).display()

today,before_7_days
2026-03-17,2026-03-10


##### 6) to calculate a deadline of 2 days 4 hours

In [0]:
data = [(1,)]
df2 = spark.createDataFrame(data, ["id"])

df2.select(
    current_timestamp().alias("start_time"),
    (current_timestamp() + make_interval(lit(0), lit(0), lit(0), lit(2), lit(4), lit(0))).alias("deadline")
).display()

start_time,deadline
2026-03-17T19:32:09.136Z,2026-03-19T23:32:09.136Z


##### 7) Using Columns Instead of Literals

In [0]:
data = [
    (1, 2, 5),   # years, months, days
    (0, 1, 10)
]

df3 = spark.createDataFrame(data, ["yrs", "mths", "days"])
display(df3)

df3.select("*",
    make_interval("yrs", "mths", "days", lit(0), lit(0), lit(0)).alias("dynamic_interval")
).display()

yrs,mths,days
1,2,5
0,1,10


yrs,mths,days,dynamic_interval
1,2,5,1 years 2 months 35 days
0,1,10,1 months 70 days
